# D.R.O.N.A. — Full Pipeline Demo

End-to-end walkthrough of the Phase 1 pipeline from student query to robot gesture.

**No ROS2 required.** Everything runs in-process.

**Sections:**
1. Bias detection on sample queries
2. RAG retrieval from ChromaDB
3. Advising engine with mocked LLM (no Ollama needed)
4. Full advising engine with real LLM (Ollama required)
5. Session machine lifecycle walk-through
6. Gesture execution and visualisation
7. C4 Nepal citation ratio measurement

In [ ]:
import sys
sys.path.insert(0, '..')

import uuid
import warnings
warnings.filterwarnings('ignore')

from drona.utils.logging import setup_logging
setup_logging('WARNING')

from drona.contracts import (
    AdvisingQuery, StudentProfile, DataTier, GestureType, InteractionAction
)

def make_query(text: str) -> AdvisingQuery:
    return AdvisingQuery(
        query_id=str(uuid.uuid4()),
        query_text=text,
        profile=StudentProfile(
            session_id=str(uuid.uuid4()),
            year_of_study=3,
            declared_skills=['Python', 'SQL', 'Linux'],
        ),
    )

print('Imports OK.')

## 1. Bias Detection

In [ ]:
from drona.advising.bias_detector import BiasDetector

bd = BiasDetector()

test_queries = [
    'What career paths suit a Python developer in Nepal?',
    'I only want to work at Google, nowhere else will do.',
    'My friend just got into ML and is earning a lot, I should do ML too.',
    'I told everyone I would be a data scientist. I cannot change now.',
    'My parents insist I work at a bank. I cannot disappoint them.',
]

print('Query → Detected Bias')
print('─' * 70)
for q in test_queries:
    flags = bd.detect(q)
    bias_str = ', '.join(f.bias_type for f in flags) if flags else 'none'
    print(f'{q[:55]:<55} → {bias_str}')

## 2. RAG Retrieval

In [ ]:
from drona.advising.retriever import Retriever

retriever = Retriever()

query_text = 'Software developer jobs in Kathmandu for Python developers'
docs = retriever.retrieve_raw(query_text, top_k=5)

if docs:
    print(f'Retrieved {len(docs)} documents for: "{query_text}"\n')
    for i, d in enumerate(docs, 1):
        tier = d.metadata.get('tier', '?')
        src  = d.metadata.get('source_type', '?')
        print(f'[{i}] tier={tier:15s} src={src:15s} rrf={d.rrf_score:.4f}')
        print(f'    {d.text[:120]}...')
        print()
    
    nepal_frac = sum(1 for d in docs if d.metadata.get('tier') == 'nepal') / len(docs)
    print(f'Nepal fraction: {nepal_frac:.0%}')
else:
    print('No documents retrieved. Ensure ChromaDB is populated:')
    print('  python scripts/ingest_data.py')

## 3. AdvisingEngine — Mocked LLM (no Ollama needed)

In [ ]:
from unittest.mock import MagicMock
from drona.advising.engine import AdvisingEngine
from drona.advising.retriever import _Doc
from drona.contracts import PathwayRecommendation

# Mock retriever returns synthetic Nepal-tier docs
mock_docs = [
    _Doc(id=f'doc{i}', text=f'Senior Python developer at Leapfrog Technology, Kathmandu. Salary NPR 80,000.',
         metadata={'source_type': 'job_posting', 'tier': 'nepal'}, rrf_score=0.6 - i*0.05)
    for i in range(5)
]
mock_retriever = MagicMock()
mock_retriever.retrieve_raw.return_value = mock_docs

mock_reranker = MagicMock()
mock_reranker.rerank_docs.return_value = mock_docs

mock_llm = MagicMock()
mock_llm.generate.return_value = (
    [
        PathwayRecommendation(pathway_title='Software Developer — Nepal Fintech', rationale='Strong Python demand in Nepal fintech sector.', confidence='high'),
        PathwayRecommendation(pathway_title='Data Engineer — Remote Nepal', rationale='Growing remote data engineering roles.', confidence='medium'),
        PathwayRecommendation(pathway_title='Cloud Engineer — Kathmandu', rationale='AWS/GCP adoption growing in Kathmandu enterprises.', confidence='medium'),
    ],
    'You have strong options as a Python developer in Nepal.',
    False, None,
)

engine = AdvisingEngine(retriever=mock_retriever, reranker=mock_reranker, llm=mock_llm)

query = make_query('What career paths are available for a Python developer in Nepal?')
response = engine.advise(query)

print(f'Summary: {response.summary}')
print(f'Refusal: {response.refusal}')
print(f'Bias flags: {[f.bias_type for f in response.bias_flags]}')
print(f'Generation time: {response.generation_time_ms} ms')
print()
for i, pw in enumerate(response.pathways, 1):
    print(f'[{i}] {pw.pathway_title} ({pw.confidence})')
    print(f'    {pw.rationale}')

## 4. AdvisingEngine — Real LLM (requires Ollama)

Start Ollama first: `ollama serve && ollama pull mistral`

In [ ]:
import requests

def check_ollama() -> bool:
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        return r.status_code == 200
    except Exception:
        return False

OLLAMA_AVAILABLE = check_ollama()
print(f'Ollama: {"✅ running" if OLLAMA_AVAILABLE else "❌ not running (start with: ollama serve)"}')

CHROMADB_HAS_DOCS = len(docs) > 0
print(f'ChromaDB: {"✅ populated" if CHROMADB_HAS_DOCS else "❌ empty (run: python scripts/ingest_data.py)"}')

In [ ]:
if OLLAMA_AVAILABLE and CHROMADB_HAS_DOCS:
    import time
    print('Running full AdvisingEngine...')
    real_engine = AdvisingEngine()  # uses real ChromaDB + Ollama
    
    query = make_query('What career paths are available for a Python developer in Nepal?')
    t0 = time.monotonic()
    response = real_engine.advise(query)
    elapsed = (time.monotonic() - t0) * 1000
    
    print(f'Total wall time: {elapsed:.0f} ms  |  Engine-reported: {response.generation_time_ms} ms')
    print(f'Refusal: {response.refusal}')
    if response.refusal:
        print(f'Reason: {response.refusal_reason}')
    else:
        print(f'Speak text: {response.speak_text}')
        print(f'Bias flags: {[f.bias_type for f in response.bias_flags]}')
        nepal_cits = sum(1 for c in [cit for pw in response.pathways for cit in pw.citations]
                         if c.tier == DataTier.NEPAL)
        total_cits = sum(len(pw.citations) for pw in response.pathways)
        print(f'Nepal citations: {nepal_cits}/{total_cits} ({nepal_cits/total_cits:.0%})' if total_cits else 'No citations')
        for i, pw in enumerate(response.pathways, 1):
            print(f'  [{i}] {pw.pathway_title} ({pw.confidence})')
else:
    print('Skipped — Ollama or ChromaDB not available.')

## 5. Session Machine Lifecycle

In [ ]:
from drona.orchestrator.session_machine import SessionMachine, SessionState
from drona.perception.mediapipe_detector import StudentDetection, EngagementState

machine = SessionMachine(timeout_s=60)

def det(eng, conf=0.9, dist=1.2):
    return StudentDetection(detection_id=str(uuid.uuid4()), engagement=eng, confidence=conf, estimated_distance_m=dist)

transitions = []
def step(label, detection=None, action=None):
    prev = machine.state
    if detection:
        machine.feed_detection(detection)
    if action:
        action()
    curr = machine.state
    changed = curr != prev
    transitions.append((label, prev.value, curr.value, changed))
    arrow = '→' if changed else '─'
    print(f'{label:30s}: {prev.value:20s} {arrow} {curr.value}')

print('Session lifecycle:')
step('Student approaches',         det(EngagementState.APPROACHING, 0.6))
step('Student engages',            det(EngagementState.ENGAGED, 0.9))
step('Greet gesture done',         action=machine.mark_greeted)
step('Student submits query',      action=lambda: machine.submit_query('What jobs suit a Python dev?'))
step('Response delivered',         action=machine.mark_response_delivered)
step('Session closed',             action=machine.mark_session_closed)
print(f'\nFinal state: {machine.state.value}')
print(f'New session ID: {machine.session_id}')
print(f'Summary: {machine.session_summary()}')

## 6. Gesture Execution

In [ ]:
from drona.interaction.gesture_dispatcher import GestureDispatcher
from drona.contracts import InteractionAction, GestureType

dispatcher = GestureDispatcher(checkpoint_base_dir=None)

print('Executing all gestures:')
for gesture_type in GestureType:
    action = InteractionAction(action_id=str(uuid.uuid4()), gesture=gesture_type)
    result = dispatcher.execute(action)
    status = '✅' if result.success else '❌'
    dur = f'{result.actual_duration_seconds:.3f}s' if result.actual_duration_seconds else 'N/A'
    print(f'  {status} {gesture_type.value:10s}: {dur}')

## 7. C4 Nepal Citation Ratio

In [ ]:
import json, os

C4_QUERIES = [
    'Python developer jobs Nepal',
    'Data science career Kathmandu BSc Computing',
    'Software engineering entry level Nepal',
    'Cloud computing roles Nepal fintech',
    'Cybersecurity analyst Nepal government',
]

nepal_fracs = []
for q_text in C4_QUERIES:
    docs = retriever.retrieve_raw(q_text, top_k=10)
    if docs:
        frac = sum(1 for d in docs if d.metadata.get('tier') == 'nepal') / len(docs)
        nepal_fracs.append(frac)
        print(f'  {q_text[:45]:<45}: {frac:.0%} Nepal ({sum(1 for d in docs if d.metadata.get("tier")=="nepal")}/{len(docs)})')
    else:
        print(f'  {q_text:<45}: no docs')

if nepal_fracs:
    mean_frac = sum(nepal_fracs) / len(nepal_fracs)
    print(f'\nMean Nepal fraction (retriever): {mean_frac:.1%}')
    print(f'C4 retriever check: {"✅ PASS" if mean_frac >= 0.6 else "❌ FAIL"} (target ≥ 60%)')
    
    os.makedirs('../data/evaluation', exist_ok=True)
    with open('../data/evaluation/c4_nepal_ratio_retriever.json', 'w') as f:
        json.dump({'mean_nepal_fraction': mean_frac, 'n_queries': len(nepal_fracs), 'per_query': nepal_fracs}, f, indent=2)
else:
    print('ChromaDB empty — populate first with: python scripts/ingest_data.py')